# [LAB] 캡스톤: 리서치 자동화 멀티 에이전트 시스템

주제를 넣으면 Agent가 자료를 모으고, 사람이 결과를 확인해 승인하면 보고서를 쓰고, 마지막에 보고서 품질을 점수로 매기는 프로그램을 완성합니다.
이 노트북에는 채울 코드가 없습니다. 코드는 `capstone/` 폴더의 `capstone/starter.py`를 AI와 함께 고치고, 터미널에서 실행합니다. 이 노트북은 순서와 확인 기준을 알려 주는 안내서입니다.

**목차**

- 1. 캡스톤 소개
  - 1.1 진행 방법
- 2. 주제 고르기
- 3. 구조도
- 4. 폴더 둘러보기
- 5. 준비: MCP 서버 켜기
- 6. 단계별 체크리스트
  - 6.1 도구 만들기
  - 6.2 State 채우기
  - 6.3 MCP 연결과 두 Agent
  - 6.4 노드 만들기
  - 6.5 그래프 조립과 승인 지점
  - 6.6 실행과 재개
  - 6.7 보고서 평가
- 7. 전체 실행
  - 7.1 Langfuse에서 내 실행 찾기
- 8. 정답 코드
- 9. 평가 기준

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env(["EMBED_PROVIDER", "RAG_PROVIDER", "LANGFUSE"])

In [ ]:
import truststore

# 파이썬이 PC에 등록된 인증서(회사 인증서 포함)를 믿게 합니다. Langfuse보다 먼저 둡니다
truststore.inject_into_ssl()

from langfuse import get_client  # noqa: E402

# Langfuse 키가 맞는지 한 번 확인합니다(키 값은 출력하지 않습니다)
print(
    "🟢 Langfuse 연결 확인"
    if get_client().auth_check()
    else "⛔ Langfuse 연결 실패: .env의 LANGFUSE_ 값 세 개를 확인하세요"
)

**볼 지점**
- "준비 완료"와 "🟢 Langfuse 연결 확인"이 나오면 다음으로 넘어갑니다.
- ⛔가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고친 뒤 다시 실행합니다. 모르겠으면 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 다른 곳에 붙여 넣지 않습니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 터미널 B에 나온 오류 메시지 전체와 방금 고친 단계(6.1~6.7 중 어디인지)를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 1. 캡스톤 소개

만들 것은 **리서치 자동화 보고서 작성 Multi-Agent 시스템**입니다. 흐름은 여섯 단계입니다.

1. 사용자가 주제를 입력합니다.
2. Supervisor가 Researcher 실행을 결정합니다.
3. Researcher가 웹 검색, RAGaaS, MCP 문서 검색으로 자료를 모읍니다.
4. 사용자가 리서치 결과를 확인하고 보고서 작성을 승인합니다.
5. Writer가 보고서 초안을 작성합니다.
6. 최종 보고서를 평가하고 결과를 출력합니다.

4일 동안 배운 기술이 모두 들어갑니다. LangGraph 멀티 에이전트(Supervisor), 사람 승인(Human-in-the-loop), MCP 서버, RAGaaS, Deep Agents의 Skill, Langfuse 추적과 점수입니다.

### 1.1 진행 방법

1. 맨 위 **환경 점검** 두 셀(접힌 코드 셀)을 실행해 설정을 확인합니다.
2. 1~4장에서 무엇을 만드는지, 폴더에 무엇이 있는지 봅니다.
3. 5장대로 터미널 A에서 MCP 서버를 켭니다.
4. 6장의 단계를 차례로 진행합니다. 단계마다 **할 일** → **프롬프트 힌트**를 보고 AI에게 지시를 직접 씀 → AI가 고친 코드를 읽고 받아들일지 판단합니다.
5. 7장대로 터미널 B에서 실행하고, 🟢/⛔ 성공 기준으로 결과를 스스로 판단합니다.
6. 9장의 평가 기준을 확인합니다. 정답 코드는 수업이 끝난 뒤 공개합니다.

AI에게 파일을 보여 줄 때는 `capstone/starter.py`를 열어 두고 지시합니다. 한 번에 한 단계만 시키면 결과를 검토하기 쉽습니다.

## 2. 주제 고르기

둘 중 하나를 고릅니다. 어느 쪽이든 6장의 체크리스트를 똑같이 통과해야 합니다.

| | A. 제공 주제 | B. 자유 주제 |
|---|---|---|
| 주제 | "생성형 AI가 바꾸는 소프트웨어 개발의 미래" | 내 업무와 관련된 주제 한 문장 |
| 문서 | `capstone/data/`의 문서 2개 그대로 | `capstone/data/`에 내 주제의 md 문서를 넣음 |
| 보고서 양식 | Writer skill 그대로 | 필요하면 `capstone/skills/writer/` 안의 양식을 내 업무에 맞게 바꿈 |
| 고칠 곳 | `capstone/starter.py`의 TODO | A와 같음 + 맨 아래 `main('…')`의 주제 문장 |

처음이라면 A를 권합니다. A를 끝낸 뒤 B로 바꿔 한 번 더 실행해도 됩니다.

## 3. 구조도

In [ ]:
from IPython.display import HTML, display

# 캡스톤 구조도(그림만 그립니다)
display(
    HTML(
        '<div style=\'color:#222\'><svg viewBox="0 0 880 420" width="100%" style="max-width:880px;font-family:sans-serif" role="img" aria-label="캡스톤 구조도"><defs><marker id="cap-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#555"/></marker></defs><rect x="0" y="0" width="880" height="420" rx="10" fill="#ffffff" stroke="#ddd"/><rect x="20" y="175" width="80" height="40" rx="20" fill="#eeeeee" stroke="#888" stroke-width="1.5"/><text x="60.0" y="200.0" text-anchor="middle" font-size="14" font-weight="bold">START</text><rect x="150" y="170" width="150" height="50" rx="8" fill="#e8f0fe" stroke="#4a74c9" stroke-width="1.5"/><text x="225.0" y="191.0" text-anchor="middle" font-size="14" font-weight="bold">Supervisor</text><text x="225.0" y="208.0" text-anchor="middle" font-size="11" fill="#444">다음 차례 결정</text><rect x="400" y="40" width="160" height="50" rx="8" fill="#e8f0fe" stroke="#4a74c9" stroke-width="1.5"/><text x="480.0" y="61.0" text-anchor="middle" font-size="14" font-weight="bold">Researcher</text><text x="480.0" y="78.0" text-anchor="middle" font-size="11" fill="#444">자료 수집</text><rect x="400" y="96" width="160" height="26" rx="6" fill="#f6f6f6" stroke="#999"/><text x="480.0" y="113.0" text-anchor="middle" font-size="11">skill: skills/researcher/</text><rect x="620" y="20" width="245" height="90" rx="6" fill="#f6f6f6" stroke="#999"/><text x="632" y="42" font-size="12" font-weight="bold">Researcher 도구</text><text x="632" y="64" font-size="11">search_web · rag_search(RAGaaS)</text><text x="632" y="84" font-size="11">MCP: list_docs · read_doc · search_docs</text><rect x="400" y="170" width="160" height="50" rx="8" fill="#fff4e5" stroke="#e0a030" stroke-width="1.5"/><text x="480.0" y="191.0" text-anchor="middle" font-size="14" font-weight="bold">⏸ 사용자 승인</text><text x="480.0" y="208.0" text-anchor="middle" font-size="11" fill="#444">y: 재개 · n: 중단</text><rect x="620" y="132" width="160" height="26" rx="6" fill="#f6f6f6" stroke="#999"/><text x="700.0" y="149.0" text-anchor="middle" font-size="11">skill: skills/writer/</text><rect x="620" y="170" width="160" height="50" rx="8" fill="#e8f0fe" stroke="#4a74c9" stroke-width="1.5"/><text x="700.0" y="191.0" text-anchor="middle" font-size="14" font-weight="bold">Writer</text><text x="700.0" y="208.0" text-anchor="middle" font-size="11" fill="#444">보고서 초안</text><rect x="400" y="280" width="160" height="40" rx="20" fill="#eeeeee" stroke="#888" stroke-width="1.5"/><text x="480.0" y="305.0" text-anchor="middle" font-size="14" font-weight="bold">END</text><rect x="150" y="350" width="150" height="50" rx="8" fill="#eef7ee" stroke="#4a9a5a" stroke-width="1.5"/><text x="225.0" y="371.0" text-anchor="middle" font-size="14" font-weight="bold">점수 출력</text><text x="225.0" y="388.0" text-anchor="middle" font-size="11" fill="#444">막대 · 종합 점수</text><rect x="400" y="350" width="160" height="50" rx="8" fill="#eef7ee" stroke="#4a9a5a" stroke-width="1.5"/><text x="480.0" y="371.0" text-anchor="middle" font-size="14" font-weight="bold">LLM Judge</text><text x="480.0" y="388.0" text-anchor="middle" font-size="11" fill="#444">4항목 0.0~1.0 채점</text><rect x="620" y="350" width="160" height="50" rx="8" fill="#eef7ee" stroke="#4a9a5a" stroke-width="1.5"/><text x="700.0" y="371.0" text-anchor="middle" font-size="14" font-weight="bold">Langfuse Score</text><text x="700.0" y="388.0" text-anchor="middle" font-size="11" fill="#444">항목별 점수 기록</text><path d="M100,195 L148,195" fill="none" stroke="#555" stroke-width="1.5" marker-end="url(#cap-arr)"/><path d="M300,178 L398,60" fill="none" stroke="#555" stroke-width="1.5" marker-end="url(#cap-arr)"/><text x="262" y="130" font-size="11" fill="#333">리서치 전</text><path d="M400,80 L306,184" fill="none" stroke="#555" stroke-width="1.5" marker-end="url(#cap-arr)"/><path d="M300,195 L398,195" fill="none" stroke="#555" stroke-width="1.5" marker-end="url(#cap-arr)"/><text x="318" y="188" font-size="11" fill="#333">리서치 후</text><path d="M560,195 L618,195" fill="none" stroke="#555" stroke-width="1.5" marker-end="url(#cap-arr)"/><text x="584" y="188" font-size="11" fill="#333">y</text><path d="M560,65 L618,65" fill="none" stroke="#555" stroke-width="1.5" stroke-dasharray="4 3" marker-end="url(#cap-arr)"/><path d="M700,220 L700,250 L260,250 L260,222" fill="none" stroke="#555" stroke-width="1.5" marker-end="url(#cap-arr)"/><text x="420" y="244" font-size="11" fill="#333">초안 후 Supervisor로 복귀</text><path d="M195,220 L195,300 L398,300" fill="none" stroke="#555" stroke-width="1.5" marker-end="url(#cap-arr)"/><text x="250" y="294" font-size="11" fill="#333">초안까지 끝</text><path d="M480,320 L480,348" fill="none" stroke="#555" stroke-width="1.5" marker-end="url(#cap-arr)"/><path d="M560,375 L618,375" fill="none" stroke="#555" stroke-width="1.5" marker-end="url(#cap-arr)"/><path d="M400,375 L302,375" fill="none" stroke="#555" stroke-width="1.5" marker-end="url(#cap-arr)"/></svg></div>'
    )
)

**그림 읽는 법**
- 실선 화살표는 그래프가 실행되는 순서입니다. Supervisor는 매번 "리서치 전 → Researcher, 리서치 후 → (승인) → Writer, 초안까지 끝 → END"를 고릅니다.
- Researcher와 Writer는 일을 마치면 Supervisor로 돌아갑니다.
- ⏸ 사용자 승인은 Writer 바로 앞에서 그래프를 멈추는 자리입니다.
- 점선은 Researcher가 쓰는 도구입니다. 회색 상자는 각 Agent가 읽는 skill 폴더입니다.
- END 뒤에 LLM Judge가 보고서를 채점하고, 점수를 화면과 Langfuse에 남깁니다.

## 4. 폴더 둘러보기

`capstone/` 폴더의 파일과 흐름에서의 위치입니다.

| 파일 | 하는 일 | 흐름에서 위치 |
|---|---|---|
| `capstone/starter.py` | 전체 프로그램 뼈대. `TODO` 자리를 채웁니다 | 흐름 1~6 전부 |
| `capstone/mcp_server.py` | `data/`의 문서를 도구 3개로 내주는 MCP 서버. 그대로 씁니다 | 흐름 3(자료 수집) |
| `capstone/skills/researcher/research/SKILL.md` | Researcher 규칙: 도구 쓰는 순서, 리서치 결과 형식(`## 리서치 결과:`) | 흐름 3 |
| `capstone/skills/writer/ds-report-writer/SKILL.md` | Writer 규칙: `[DS-AI-RPT v2.1]` 보고서 양식, TL;DR 3줄, 현황 분석 표 | 흐름 5 |
| `capstone/data/ai_trends.md`, `capstone/data/llm_market.md` | MCP 서버가 보여 주는 문서 | 흐름 3 |
| `capstone/README.md` | 이 노트북 없이 폴더만 열어도 볼 수 있는 요약 | — |

`capstone/starter.py`에서 Agent, 노드, 그래프는 모두 `main()` 함수 안에 있습니다. MCP 도구를 서버에서 비동기로 받아 와야 Agent를 만들 수 있기 때문입니다.

## 5. 준비: MCP 서버 켜기

VS Code 메뉴 **터미널 → 새 터미널**로 터미널을 엽니다(저장소 맨 위 폴더에서 열립니다). 이 터미널을 **터미널 A**라고 하겠습니다. 아래 명령을 붙여 넣고 실행합니다.

```
uv run python course/Chapter02/capstone/mcp_server.py
```

이 터미널은 끝날 때까지 닫지 않습니다. 서버 파일은 고치지 않으니 한 번 켜 두면 됩니다. 다 쓴 뒤에는 **Ctrl+C**로 끕니다.

**성공 기준**

- 🟢 터미널 A에 `Uvicorn running on`과 `127.0.0.1:8001`이 든 줄이 나오고, 명령이 끝나지 않고 기다립니다.
- ⛔ `error while attempting to bind on address`가 든 `ERROR:` 줄이 나오고 명령이 끝납니다(맥은 `address already in use`, Windows는 `[Errno 10048]`) → 8001 포트를 다른 서버(3일차 A2A 서버, 앞서 켠 캡스톤 서버 등)가 쓰고 있습니다. 그 서버를 띄운 터미널에서 **Ctrl+C**로 끄고 다시 실행합니다. 이때도 위쪽에 `Starting MCP server` 줄은 나오니, 마지막 줄로 판단합니다.
- ⛔ 환경 점검 셀에서 Langfuse가 ⛔입니다 → `.env`의 Langfuse 값 세 개를 확인합니다. 그대로 진행해도 보고서와 점수는 나오지만 Langfuse에 기록되지 않습니다.

## 6. 단계별 체크리스트

`capstone/starter.py`의 `TODO` 12개를 일곱 단계(6.1~6.7)로 묶었습니다. 6.1~6.7을 모두 채운 뒤 7장에서 한 번에 실행해 확인합니다. 각 단계의 🟢/⛔는 그때 볼 화면입니다.

- 중간에 실행하면 6.6을 채우기 전에는 `⚠️ 실행 코드(6.6)가 아직 비어 있어 여기서 끝납니다.`가 나오고 끝납니다(6.3까지 채웠으면 그 위에 `MCP 도구 로드: [...]`). 6.6까지 채웠으면 보고서까지 나오고 점수는 나오지 않습니다. 아직 채우지 않은 단계 때문이니 정상입니다.
- 🟢에 적힌 `🔍 Researcher 실행 중...` 같은 출력 문구는 정답 코드가 출력하는 문구입니다. 내 코드의 문구가 달라도 같은 뜻의 줄이 나오면 통과입니다.
- AI가 함수나 클래스를 통째로 새로 쓰면 `capstone/starter.py`에 있던 같은 이름의 빈 함수·클래스가 남지 않았는지 봅니다. 같은 이름이 두 번 있으면 뒤에 있는 것이 앞의 것을 덮어씁니다.

단계를 마칠 때마다 AI가 고친 부분을 직접 읽어 봅니다. 모르는 줄이 있으면 그 줄이 무엇을 하는지 AI에게 물어봅니다.

### 6.1 도구 만들기

**할 일** `capstone/starter.py`의 `# TODO: 사용할 도구를 정의` 아래 `search_web`과, `rag_search` 안의 TODO를 채웁니다. `search_web`은 실제 검색 대신 정해 둔 검색 결과 몇 줄을 돌려주는 흉내 도구면 충분합니다. `rag_search`는 RAGaaS의 `retrieve-rrf`로 문서를 찾아 본문을 돌려줍니다. 

RAGaaS 요청 값(AI에게 그대로 알려 줍니다): `index_name`(인덱스 이름, `.env`의 `index_name` 값), `permission_groups`(`["rag-public"]`), `query_text`(검색어), `num_result_doc`(결과 개수, 예: 3), `fields_exclude`(`["v_merge_title_content"]`). 요청은 `config.ragaas`의 `post_ragaas("retrieve-rrf", payload)`로 보내고, 본문은 응답 JSON의 `hits` → `hits` 목록 → `_source` → `merge_title_content`에 있습니다.

**프롬프트 힌트** 두 도구가 검색어를 받아 글(문자열)을 돌려줘야 한다고 먼저 말합니다. `rag_search`는 RAGaaS 요청에 넣을 값(인덱스 이름, 검색어, 결과 개수)과 응답에서 꺼낼 본문 필드를 AI에게 알려 줍니다. `search_web`은 실제 검색 없이 정해 둔 결과를 돌려주면 된다고 덧붙입니다.

**성공 기준**

- 🟢 7장 실행 뒤 Langfuse에서 먼저 생긴 `LangGraph` trace(Researcher 실행, 7.1 표)를 열면 `rag_search` 도구 결과가 보입니다. 사내망 밖에서는 `[MOCK RAGaaS] 질문: …`으로 시작하는 문장, 사내망에서는 실제 문서 본문입니다.
- 🟢 같은 trace의 마지막 출력(리서치 결과) 맨 아래 `### 출처`에 RAG 검색과 웹 검색이 적혀 있습니다(도구 이름 `rag_search`·`search_web` 대신 "RAG 검색"·"웹 검색"처럼 한글로 적히기도 합니다). 정답 코드는 터미널에 리서치 결과의 앞 500자만 보여 줍니다.
- ⛔ 도구 결과가 `null`입니다 → 함수 끝에 `return`이 있는지, 뒤에 같은 이름의 빈 함수가 남아 있지 않은지 봅니다.
- ⛔ `KeyError: 'hits'`가 납니다 → 응답에서 본문을 꺼내는 순서를 AI와 다시 확인합니다.
- ⛔ 사내망에서 인증 오류(401)가 납니다 → 저장소 맨 위에서 `uv run python -m config.check`로 RAGaaS 키 이름을 확인합니다.

### 6.2 State 채우기

**할 일** `# TODO: 필요한 필드를 추가하세요.` 아래 `State`에 노드끼리 주고받을 값을 더합니다. `main()` 안의 `initial_state`에도 같은 이름으로 처음 값을 넣습니다.

**프롬프트 힌트** 노드들이 서로 넘겨받아야 하는 값(주제, 리서치 결과, 보고서 초안, 다음 차례)을 흐름 순서대로 적어 AI에게 보여 줍니다. `initial_state`에도 같은 이름으로 처음 값을 넣어야 한다고 말합니다.

**성공 기준**

- 🟢 7장 실행 중에 `KeyError`가 나지 않고 `🔍 Researcher 실행 중...`까지 진행됩니다.
- ⛔ `KeyError: 'topic'`처럼 이름 오류가 납니다 → `State`와 `initial_state`의 이름이 노드에서 쓰는 이름과 같은지 봅니다.

### 6.3 MCP 연결과 두 Agent

**할 일** `main()` 안의 세 TODO를 채웁니다. MCP 서버에 연결해 도구를 받아 오고(`# TODO: MCP 서버에 연결해 …`), 자료조사 skill을 가진 Researcher와 보고서 작성 skill을 가진 Writer를 만듭니다.

**프롬프트 힌트** MCP 도구는 8001 포트의 서버(`localhost:8001/sse`, SSE 방식)에서 비동기로 받아 온다고 알려 줍니다. 각 Agent에 줄 도구, skill 폴더(`skills/researcher/`, `skills/writer/`), 역할 밖의 일을 하지 않게 하는 system prompt를 AI에게 알려 줍니다. 두 Agent 모두 실행 상태를 기억할 체크포인터를 붙인다고 말합니다.

**성공 기준**

- 🟢 7장 실행 첫 줄에 `MCP 도구 로드: ['list_docs', 'read_doc', 'search_docs']`가 나옵니다.
- 🟢 리서치 결과가 `## 리서치 결과:`로 시작하고 `### 핵심 발견` 같은 소제목이 있습니다(Researcher skill 형식).
- ⛔ `ConnectError: All connection attempts failed`가 납니다 → 5장의 MCP 서버를 먼저 켭니다.
- ⛔ 리서치 결과가 `## 리서치 결과:` 형식이 아닙니다 → skill 폴더를 못 찾은 것입니다. skill 경로는 `SKILL.md`가 든 폴더의 한 단계 위(`skills/researcher/`)를 가리켜야 합니다.
- ⛔ `await` 관련 오류가 납니다 → MCP 도구를 받아 오는 줄과 Agent를 만드는 줄이 `main()` 안에 있는지 봅니다.

### 6.4 노드 만들기

**할 일** `# TODO: Supervisor 노드 정의`, `# TODO: Researcher 노드 정의`, `# TODO: Writer 노드 정의` 아래 세 함수의 `pass`를 채웁니다.

**프롬프트 힌트** Supervisor에게 지금 상황(주제, 리서치 완료 여부, 초안 완료 여부)을 알려 주고, 세 선택지(researcher, writer, END)를 언제 고르는지 먼저 정리해 AI에게 줍니다. Researcher·Writer 노드는 Agent 실행 결과의 마지막 메시지를 State의 어느 값에 넣는지 말합니다. 두 노드는 비동기(`async`)이고 Langfuse 추적 설정을 함께 넘기며, 시작할 때 `🔍 Researcher 실행 중...`·`✍️  Writer 실행 중...`을 출력한다고 덧붙입니다.

**성공 기준**

- 🟢 7장 실행에서 `🔍 Researcher 실행 중...`과 `✍️  Writer 실행 중...`이 한 번씩만 나옵니다.
- ⛔ 같은 Agent가 되풀이해 실행됩니다 → 터미널 B에서 **Ctrl+C**로 멈춥니다. Supervisor가 받는 상황 글에 리서치·초안 완료 여부가 들어 있는지, 두 노드가 결과를 State 값으로 돌려주는지(정답은 `return {'research_result': …}`·`return {'draft': …}`) 봅니다.
- ⛔ Writer가 리서치 결과 없이 글을 씁니다 → Writer 노드가 State의 리서치 결과를 지시에 넣는지 봅니다.
- ⛔ `KeyError: ''`가 납니다 → Supervisor 노드가 다음 차례를 돌려주지 않은 것입니다. `pass`만 있는 같은 이름의 빈 함수가 뒤에 남아 있지 않은지 봅니다.

### 6.5 그래프 조립과 승인 지점

**할 일** `# TODO: 그래프 조립` 아래에 노드 사이 연결을 넣고, 바로 아래 `app = builder.compile()`에 Writer 앞에서 멈추는 설정을 더합니다. 시작 연결 한 줄(`builder.add_edge(START, 'supervisor')`)은 빈 상태로 실행해도 멈추지 않게 미리 넣어 둔 자리표시입니다. 그 아래에 나머지 연결을 이어서 넣습니다.

**프롬프트 힌트** 노드 사이 연결(시작 → Supervisor, Supervisor의 세 갈래, 각 Agent 뒤 Supervisor로 복귀)을 그림처럼 말로 적어 AI에게 줍니다. Writer 실행 전에 멈춰야 한다는 조건과, 멈춘 상태를 기억할 장치가 필요하다는 점을 함께 말합니다.

**성공 기준**

- 🟢 7장 실행에서 리서치 결과가 나온 뒤 `보고서 작성을 시작할까요? (y/n):`에서 멈춥니다.
- ⛔ 멈추지 않고 곧장 Writer까지 갑니다 → `compile()`에 체크포인터와 Writer 앞 중단 설정이 있는지 봅니다.
- ⛔ `KeyError: 'END'` 같은 갈림길 오류가 납니다 → Supervisor의 세 선택지가 모두 연결되어 있는지 봅니다.

### 6.6 실행과 재개

**할 일** `main()` 맨 아래 `# TODO: app.ainvoke로 실행하고 단계별 진행 상황 출력`을 채웁니다. 그 아래 `⚠️ 실행 코드(6.6)가 아직 비어 있어…`를 출력하는 자리표시 줄은 지웁니다. 리서치까지 실행 → 결과 보여 주기 → 터미널에서 승인 입력 → Writer부터 이어서 실행 → 보고서 출력 순서입니다.

**프롬프트 힌트** 리서치까지 실행 → 결과 보여 주기 → `y/n` 입력 받기 → 재개의 순서를 AI에게 알려 줍니다. 재개할 때는 같은 `thread_id`로, 새 입력 없이 이어서 실행한다고 말합니다. `n`이면 안내 문구를 출력하고 끝낸다고 덧붙입니다.

**성공 기준**

- 🟢 `y`를 입력하면 `--- 2단계: 보고서 작성 ---` 뒤에 `[DS-AI-RPT v2.1]`로 시작하는 보고서가 나옵니다. TL;DR이 세 줄(①②③)이고 현황 분석에 표가 있습니다.
- 🟢 `n`을 입력하면 `취소됐습니다.`가 나오고 끝납니다. 한 번 실행에서는 한쪽만 볼 수 있으니, 다른 쪽은 한 번 더 실행해 확인합니다.
- ⛔ 승인한 뒤 아무것도 나오지 않습니다 → 재개할 때 같은 `thread_id`와 빈 입력(`None`)으로 실행하는지 봅니다. Writer 노드가 `draft`를 돌려주는지도 봅니다.
- ⛔ `No synchronous function provided` 오류가 납니다 → 노드가 비동기라 `invoke` 대신 `ainvoke`로 실행합니다.
- ⛔ 보고서가 `[DS-AI-RPT v2.1]` 양식이 아닙니다 → 6.3의 Writer skill 경로를 봅니다.
- ⛔ `y`를 넣었는데 `취소됐습니다.`가 나옵니다 → 한글 입력 상태라 `ㅛ`가 들어갔거나, `yes`나 빈 Enter를 넣은 것입니다. 정답 코드는 `y` 한 글자만 승인으로 봅니다. 영문 입력으로 바꾸고 다시 실행합니다.

### 6.7 보고서 평가

**할 일** `evaluate_report` 함수의 `# TODO: 평가를 진행하고 Langfuse에 점수를 기록`을 채우고, `main()`에서 보고서를 출력한 뒤 이 함수로 채점해 점수를 보여 줍니다.

**프롬프트 힌트** 네 평가 항목(Relevance, Faithfulness, Completeness, Clarity)의 뜻과 0.0~1.0 점수 범위를 AI에게 그대로 알려 주고, 근거 없이 높은 점수를 주지 않게 하라고 말합니다. 점수는 채점한 호출의 trace에 항목 이름으로 하나씩 남긴다고 말합니다. 화면에는 항목별 막대와 점수·근거를 출력하고, 그 아래에 평균을 `종합 점수`로 출력한다고 덧붙입니다.

**성공 기준**

- 🟢 관련성·근거충실도·완성도·명확성 네 줄에 막대(`█░`), 점수, 근거가 나오고 마지막에 `종합 점수`가 나옵니다.
- 🟢 Langfuse에 채점 trace가 생기고 relevance·faithfulness·completeness·clarity 점수 네 개가 붙습니다(찾는 법은 7장).
- ⛔ `'NoneType' object has no attribute 'relevance'` 오류가 납니다 → `evaluate_report` 끝에 평가 결과를 돌려주는 `return`이 있는지 봅니다.
- ⛔ 점수는 나오는데 Langfuse에 없습니다 → 환경 점검 셀의 Langfuse 결과와, 프로그램 끝에서 기록을 내보내는지(`flush`) 봅니다.
- ⛔ 네 점수가 모두 1.0입니다 → Judge 지시에 엄격하게 채점하라는 조건이 있는지 봅니다.

## 7. 전체 실행

VS Code 메뉴 **터미널 → 새 터미널**로 터미널을 하나 더 엽니다(**터미널 B**). 터미널이 둘이 되면 터미널 창 오른쪽에 목록이 생기고, 눌러서 A와 B를 오갑니다. 터미널 A의 MCP 서버는 켜 둔 채로, 터미널 B에서 실행합니다.

```
uv run python course/Chapter02/capstone/starter.py
```

1. `MCP 도구 로드: [...]`가 나오고 `🔍 Researcher 실행 중...`으로 리서치가 시작됩니다.
2. 리서치 결과(앞부분)가 나온 뒤 `보고서 작성을 시작할까요? (y/n):`에서 멈춥니다. 결과를 읽어 봅니다.
3. 영문 입력 상태에서 `y`를 입력하고 Enter를 누르면 보고서 작성과 평가가 이어집니다. `n`이면 여기서 끝납니다. 한글 입력 상태의 `ㅛ`나 빈 Enter도 취소로 처리됩니다.
4. 보고서와 네 항목 점수, `종합 점수`가 나오면 끝입니다. 한 번 실행에 1분 안팎이 걸립니다.

프로그램이 끝나면 멈춰 둔 상태도 사라집니다. `y`와 `n`의 결과를 둘 다 보려면 한 번 더 실행합니다. 코드를 고친 뒤에도 터미널 B에서 같은 명령으로 다시 실행합니다. 터미널 A의 서버는 다시 켤 필요가 없습니다.

### 7.1 Langfuse에서 내 실행 찾기

Langfuse 대시보드의 Traces 목록은 최근 실행이 위에 옵니다. 방금 실행한 시각의 trace 세 개를 찾습니다.

| trace 이름 | 무엇 | 볼 것 |
|---|---|---|
| `LangGraph` (먼저 생긴 것) | Researcher 실행 | 도구 호출(`list_docs`, `read_doc`, `rag_search`, `search_web`)과 그 결과 |
| `LangGraph` (나중에 생긴 것) | Writer 실행 | skill 파일을 읽고 보고서를 쓴 과정 |
| `RunnableSequence` | LLM Judge 채점 | **Scores**에 relevance·faithfulness·completeness·clarity 네 점수 |

같은 Langfuse 프로젝트를 여럿이 함께 쓰면 다른 사람의 trace도 섞여 보입니다. 실행한 시각으로 내 것을 고릅니다.

**성공 기준**

- 🟢 `y`로 끝까지 실행한 뒤, 방금 실행한 시각에 위 세 trace가 있고 채점 trace에 점수 네 개가 붙어 있습니다.
- ⛔ trace가 하나도 없습니다 → 환경 점검 셀의 Langfuse 결과를 다시 봅니다. ⛔이면 `.env`의 Langfuse 값을 고치고 7장을 다시 실행합니다.

## 8. 정답 코드

정답 코드는 수업이 끝난 뒤에 공개합니다. 지금은 7장 실행 결과(`[DS-AI-RPT v2.1]` 보고서와 네 항목 점수)와 6장의 🟢/⛔ 기준으로 스스로 확인합니다.

## 9. 평가 기준

1. **체크리스트 통과:** 6장의 일곱 단계를 모두 통과합니다(7장 실행에서 각 단계의 🟢를 확인).
2. **보고서 품질:** LLM Judge 네 항목으로 평가합니다. 각 항목 0.0~1.0 점수와 채점 근거, 평균 점수가 나오고 Langfuse Score로 기록됩니다.

| 항목 | 화면 이름 | 보는 것 |
|---|---|---|
| Relevance | 관련성 | 주제와 관련 있는가 |
| Faithfulness | 근거충실도 | 리서치 근거에 충실한가 |
| Completeness | 완성도 | 보고서 구성이 완성되어 있는가 |
| Clarity | 명확성 | 문장과 구조가 명확한가 |

**일찍 끝났다면(선택):** 터미널의 `y/n` 승인을 Streamlit 화면의 버튼으로 바꿔 봅니다. 또는 2장의 B(자유 주제)로 바꿔 한 번 더 실행합니다.

다 쓴 뒤에는 터미널 A에서 **Ctrl+C**로 MCP 서버를 끕니다. 터미널 B의 프로그램은 점수를 출력하면 저절로 끝납니다.